# 02 · Leer la estructura del HTML

**35 minutos.**

En la notebook anterior conseguiste el HTML: un chorizo de texto con etiquetas
mezcladas. BeautifulSoup lo convierte en algo que se puede recorrer.

> **Celda de reanudación.** Si recién llegás al Meet, o si se te reinició el
> kernel, corré esta celda y seguí desde acá.

In [ ]:
from pathlib import Path

from bs4 import BeautifulSoup

DATOS = Path("datos")
ARCHIVO_ODISEA = DATOS / "odisea.html"

print("¿Tengo el archivo de la Odisea?:", ARCHIVO_ODISEA.exists())

Si dice `False`, volvé a la notebook `01` y corré la celda que descarga y guarda
la página.

## Empecemos con un HTML mínimo

Antes de meternos con una página real, uno chiquito escrito a mano. Todo lo que
hay que entender de HTML está acá adentro.

In [ ]:
html_chico = """
<html>
  <head>
    <title>Mi página de ejemplo</title>
  </head>
  <body>
    <h1>Título principal</h1>
    <p>Este es un párrafo.</p>
    <p>Y este es otro párrafo.</p>
    <a href="https://ejemplo.com">Un enlace</a>
  </body>
</html>
"""

sopa = BeautifulSoup(html_chico, "html.parser")

> En la documentación y en cualquier tutorial en inglés esta variable se llama
> `soup`. Es lo mismo; acá la llamamos `sopa` para que el código se lea en el
> mismo idioma en que hablamos.

## Las tres cosas que hay que saber de HTML

- **Etiqueta:** `<p>` abre, `</p>` cierra. Lo del medio es el contenido.
- **Anidamiento:** las etiquetas van una adentro de otra, como cajas dentro de cajas.
- **Atributo:** información extra dentro de la etiqueta de apertura, como el
  `href` de un enlace.

Con eso alcanza para hoy.

In [ ]:
print(sopa.prettify())

## Pedir una etiqueta

`sopa.title` te da la etiqueta entera. `.text` te da solo lo de adentro.

In [ ]:
print(sopa.title)
print(sopa.title.text)

## `find` y `find_all`

- `find("p")` → el **primer** `<p>` que encuentre.
- `find_all("p")` → una **lista** con todos.

Es la única diferencia entre las dos, y es la que más se confunde.

In [ ]:
print("El primero:", sopa.find("p").text)
print()

for parrafo in sopa.find_all("p"):
    print("-", parrafo.text)

## Atributos

Al atributo se le pide con corchetes, como a un diccionario.

In [ ]:
enlace = sopa.find("a")

print("Texto del enlace:", enlace.text)
print("A dónde apunta:  ", enlace["href"])

---

## Ahora sí, una página real

La Odisea que descargaste en la notebook anterior.

In [ ]:
html = ARCHIVO_ODISEA.read_text(encoding="utf-8")
sopa = BeautifulSoup(html, "html.parser")

print("Título:  ", sopa.title.text.strip())
print("Párrafos:", len(sopa.find_all("p")))

In [ ]:
import random

parrafos = sopa.find_all("p")
elegido = random.choice(parrafos).text.strip()

print(elegido)
print()
print(f"({len(elegido.split())} palabras)")

---

## Un error que parece inofensivo

Para contar cuántas veces aparece una palabra, es tentador escribir:

```python
str(sopa).count("Ulises")
```

Parece que funciona. No funciona: `str(sopa)` devuelve el HTML **con las etiquetas
adentro**. Estás contando dentro de los `<div>`, dentro de los `href`, dentro de
los comentarios del código.

`get_text()` devuelve solo lo que un humano vería en pantalla.

In [ ]:
con_etiquetas = str(sopa).lower()
solo_texto = sopa.get_text().lower()

print("Caracteres con etiquetas:", len(con_etiquetas))
print("Caracteres de solo texto:", len(solo_texto))
print()
print("'class' contando etiquetas:", con_etiquetas.count("class"))
print("'class' contando texto:    ", solo_texto.count("class"))

La palabra `class` aparece cientos de veces en el HTML y ninguna en la Odisea.
Toda esa diferencia es ruido que se te cuela en el corpus si contás sobre el
lugar equivocado.

In [ ]:
texto = sopa.get_text().lower()


def contar(palabra):
    print(f"'{palabra}' aparece {texto.count(palabra.lower())} veces")


contar("Ulises")
contar("Minerva")
contar("Ciclope")

> **Ojo con `count`:** cuenta *pedazos de texto*, no palabras. `contar("mar")`
> también cuenta "amargo" y "marido". Contar palabras de verdad lo vemos en la
> próxima notebook.

---

## Dónde termina BeautifulSoup

La Odisea es un caso fácil: la página es casi todo texto. Un diario, no.

En una nota de un diario, el `<p>` que buscás está mezclado con el menú, las
notas relacionadas, la publicidad, los créditos y el pie de página. Para separarlos
habría que empezar a perseguir clases y `<div>` que cambian cada vez que el sitio
se rediseña, y volver a escribir todo para cada medio.

**No vamos a hacer eso.** Para artículos hay una herramienta que ya lo resuelve, y
es la próxima notebook.

## Ejercicio (5 minutos)

Sobre la Odisea:

1. ¿Cuántos `<h2>` tiene la página?
2. Mostrá el texto del **último** párrafo. (Pista: `parrafos[-1]`)
3. Buscá tres personajes más y compará sus apariciones. ¿Alguno te sorprendió?